# AIE425 — Intelligent Recommender Systems
## Lab 3: Synthetic Rating Matrices

**Name:** Ahmed Hisham  
**ID:** 223106291

In [ ]:
import random
import math
import csv
import os
import matplotlib.pyplot as plt

random.seed(425)
OUTPUT_DIR = "Activity_3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_USERS = 5000
N_ITEMS = 100
DISTRIBUTIONS = ("uniform", "normal", "poisson")


## Part 2 — Build the three samplers

In [ ]:
def sample_uniform(a, b):
    """One sample from Uniform[a, b), using random.random()."""
    u = random.random()
    return a + (b - a) * u


def sample_normal(mu, sigma):
    """One normal sample using the Box–Muller transform."""
    u1 = 1.0 - random.random()  # strictly greater than 0
    u2 = random.random()
    z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    return mu + sigma * z


def sample_poisson(lam):
    """One Poisson sample using Knuth's algorithm."""
    threshold = math.exp(-lam)
    k = 0
    product = 1.0
    while True:
        product *= random.random()
        if product <= threshold:
            return k
        k += 1


In [ ]:
# Check sample mean and population variance over 100,000 draws.
# These checks consume random draws; the seed at the top makes them reproducible.
def mean_and_population_variance(values):
    mean = sum(values) / len(values)
    variance = sum((x - mean) ** 2 for x in values) / len(values)
    return mean, variance

sample_count = 100_000
uniform_check = [sample_uniform(0, 1) for _ in range(sample_count)]
normal_check = [sample_normal(3, 1) for _ in range(sample_count)]
poisson_check = [sample_poisson(2) for _ in range(sample_count)]

print("Uniform U[0,1): mean, variance =", mean_and_population_variance(uniform_check))
print("Normal N(3,1): mean, variance =", mean_and_population_variance(normal_check))
print("Poisson(lambda=2): mean, variance =", mean_and_population_variance(poisson_check))


## Part 3 — Required rating-generation functions

In [ ]:
def generate_ratings(n, dist, **params):
    """Generate n integer ratings in [1, 5] for one user."""
    ratings = []
    for _ in range(n):
        if dist == "uniform":
            raw = sample_uniform(0.5, 5.5)
        elif dist == "normal":
            mu = params.get("mu", 3)
            sigma = params.get("sigma", 1)
            raw = sample_normal(mu, sigma)
        elif dist == "poisson":
            lam = params.get("lam", 2)
            raw = 1 + sample_poisson(lam)
        else:
            raise ValueError("dist must be 'uniform', 'normal', or 'poisson'")

        rating = int(round(raw))
        rating = max(1, min(5, rating))
        ratings.append(rating)
    return ratings


def assign_ratings(n_items, dist, max_fill=0.5, **params):
    """Create one sparse user row; None means the rating is missing."""
    if not 0 < max_fill <= 1:
        raise ValueError("max_fill must be in (0, 1].")
    max_k = int(max_fill * n_items)
    if max_k < 5:
        raise ValueError("max_fill * n_items must be at least 5.")

    # Choose k uniformly from the allowed integer range, using random.random().
    k = 5 + int(random.random() * (max_k - 5 + 1))
    k = min(k, max_k)
    chosen_items = random.sample(range(n_items), k)
    values = generate_ratings(k, dist, **params)

    row = [None] * n_items
    for item_idx, rating in zip(chosen_items, values):
        row[item_idx] = rating

    assert sum(value is not None for value in row) <= 0.5 * n_items
    return row


def build_matrix(n_users, n_items, dist, **params):
    return [assign_ratings(n_items, dist, **params) for _ in range(n_users)]


## Build and save the three 5,000 × 100 matrices

In [ ]:
# Continue from the seeded random stream above; no other RNG is used.
matrices = {}
for dist in DISTRIBUTIONS:
    matrices[dist] = build_matrix(N_USERS, N_ITEMS, dist)

    # Verify dimensions, rating values, and at least 50% missing per user.
    assert len(matrices[dist]) == N_USERS
    assert all(len(row) == N_ITEMS for row in matrices[dist])
    for row in matrices[dist]:
        observed = [x for x in row if x is not None]
        assert len(observed) <= N_ITEMS * 0.5
        assert all(isinstance(x, int) and 1 <= x <= 5 for x in observed)

    csv_path = os.path.join(OUTPUT_DIR, f"{dist}_ratings.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["user_id", "item_id", "rating"])
        for user_idx, row in enumerate(matrices[dist]):
            for item_idx, rating in enumerate(row):
                if rating is not None:
                    writer.writerow([user_idx, item_idx, rating])

    print(f"{dist}: saved {csv_path}")


## Part 4 — Sparsity, means, and homogeneity

In [ ]:
def observed_values(matrix):
    return [rating for row in matrix for rating in row if rating is not None]


def global_mean(matrix):
    values = observed_values(matrix)
    return sum(values) / len(values) if values else float("nan")


def user_means(matrix):
    means = []
    for row in matrix:
        values = [x for x in row if x is not None]
        if values:
            means.append(sum(values) / len(values))
    return means


def item_means(matrix):
    means = []
    n_items = len(matrix[0])
    for item_idx in range(n_items):
        values = [row[item_idx] for row in matrix if row[item_idx] is not None]
        if values:
            means.append(sum(values) / len(values))
        else:
            means.append(float("nan"))
    return means


def sparsity(matrix):
    total = len(matrix) * len(matrix[0])
    observed = sum(x is not None for row in matrix for x in row)
    return 1.0 - observed / total


def population_std(values):
    if not values:
        return float("nan")
    avg = sum(values) / len(values)
    return math.sqrt(sum((x - avg) ** 2 for x in values) / len(values))


def normalized_entropy(values):
    counts = [0] * 5
    for x in values:
        counts[x - 1] += 1
    total = len(values)
    probs = [count / total for count in counts if count > 0]
    entropy = -sum(p * math.log(p) for p in probs)
    return entropy / math.log(5)


def chi_square_uniform(values):
    counts = [0] * 5
    for x in values:
        counts[x - 1] += 1
    total = len(values)
    expected = total / 5
    return sum((count - expected) ** 2 / expected for count in counts)


def gini_nonnegative(values):
    """Gini coefficient for nonnegative item-popularity counts."""
    if not values or sum(values) == 0:
        return float("nan")
    sorted_values = sorted(values)
    n = len(sorted_values)
    weighted_sum = sum((i + 1) * value for i, value in enumerate(sorted_values))
    return (2 * weighted_sum) / (n * sum(sorted_values)) - (n + 1) / n


def homogeneity_metrics(matrix):
    values = observed_values(matrix)
    n_users, n_items = len(matrix), len(matrix[0])
    user_fill = [sum(x is not None for x in row) / n_items for row in matrix]
    popularity = [
        sum(row[j] is not None for row in matrix)
        for j in range(n_items)
    ]
    u_means = user_means(matrix)
    i_means = [x for x in item_means(matrix) if not math.isnan(x)]
    return {
        "Sparsity": sparsity(matrix),
        "Global mean": global_mean(matrix),
        "Mean of user means": sum(u_means) / len(u_means),
        "Smallest user mean": min(u_means),
        "Largest user mean": max(u_means),
        "Max row fill": max(user_fill),
        "Normalized entropy": normalized_entropy(values),
        "Chi-square vs uniform": chi_square_uniform(values),
        "Std of user means": population_std(u_means),
        "Std of item means": population_std(i_means),
        "Gini of item popularity": gini_nonnegative(popularity),
    }


metrics = {dist: homogeneity_metrics(matrices[dist]) for dist in DISTRIBUTIONS}
metric_order = [
    "Sparsity", "Global mean", "Mean of user means",
    "Smallest user mean", "Largest user mean", "Max row fill",
    "Normalized entropy", "Chi-square vs uniform",
    "Std of user means", "Std of item means", "Gini of item popularity"
]

print(f"{'Metric':30s} {'Uniform':>14s} {'Normal (3,1)':>14s} {'Poisson (λ=2)':>16s}")
for metric in metric_order:
    print(f"{metric:30s} {metrics['uniform'][metric]:14.5f} "
          f"{metrics['normal'][metric]:14.5f} {metrics['poisson'][metric]:16.5f}")


### Histograms and heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, dist in zip(axes, DISTRIBUTIONS):
    values = observed_values(matrices[dist])
    ax.hist(values, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
            rwidth=0.85, edgecolor="black")
    ax.set_title(dist.capitalize())
    ax.set_xlabel("Rating")
    ax.set_xticks([1, 2, 3, 4, 5])
axes[0].set_ylabel("Observed count")
fig.suptitle("Observed rating histograms")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, dist in zip(axes, DISTRIBUTIONS):
    # NaN displays as the axes' bad color; set it to white for missing ratings.
    data = [[float("nan") if x is None else x for x in row]
            for row in matrices[dist][:50]]
    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="white")
    ax.imshow(data, aspect="auto", interpolation="nearest", cmap=cmap,
              vmin=1, vmax=5)
    ax.set_title(f"{dist.capitalize()} (first 50 users)")
    ax.set_xlabel("Item")
    ax.set_ylabel("User")
fig.suptitle("Rating matrix heatmaps (white = missing)")
plt.tight_layout()
plt.show()


### Part 4 mini-example (by hand)

## Part 5 — Missing-data experiment: bias or noise?

In [ ]:
def complete_matrix(n_users, n_items, dist, **params):
    return [generate_ratings(n_items, dist, **params) for _ in range(n_users)]


def hide_ratings(full, p, mechanism):
    """Return a copy with fraction p of cells hidden."""
    if not 0 <= p < 1:
        raise ValueError("p must be in [0, 1).")
    n_users = len(full)
    n_items = len(full[0])
    total = n_users * n_items
    keep_count = int(round((1.0 - p) * total))

    if mechanism == "random":
        # Every cell is equally likely to be hidden. random.sample selects
        # exactly the requested number of cells to keep.
        keep_indices = set(random.sample(range(total), keep_count))
    elif mechanism == "dislike":
        # Follow the handout literally: w = rating and key = u / w;
        # keep the largest keys. Note that this formula conflicts with the
        # handout's stated intention: dividing by rating tends to favor
        # retaining LOW ratings, not hiding them. The discrepancy is noted below.
        keyed_cells = []
        for u in range(n_users):
            for i in range(n_items):
                weight = full[u][i]
                key = random.random() / weight
                keyed_cells.append((key, u * n_items + i))
        keyed_cells.sort(reverse=True)
        keep_indices = {flat_idx for _, flat_idx in keyed_cells[:keep_count]}
    else:
        raise ValueError("mechanism must be 'random' or 'dislike'")

    result = []
    for u, row in enumerate(full):
        new_row = []
        for i, rating in enumerate(row):
            flat_idx = u * n_items + i
            new_row.append(rating if flat_idx in keep_indices else None)
        result.append(new_row)
    return result


def mean_of_nonempty_user_means(matrix):
    means = user_means(matrix)  # skips users with zero observed ratings
    return sum(means) / len(means) if means else float("nan")


# Create the full matrix and its fixed reference values.
full_normal = complete_matrix(N_USERS, N_ITEMS, "normal", mu=3, sigma=1)
reference_global = global_mean(full_normal)
reference_user_mean = mean_of_nonempty_user_means(full_normal)
print("Reference global mean:", reference_global)
print("Reference mean of user means:", reference_user_mean)


In [ ]:
def missing_data_experiment(full, hide_fractions=(0.50, 0.75, 0.90),
                           mechanisms=("random", "dislike"), trials=20):
    ref_global = global_mean(full)
    ref_user = mean_of_nonempty_user_means(full)
    rows = []

    for mechanism in mechanisms:
        for p in hide_fractions:
            global_errors = []
            user_mean_errors = []
            for _ in range(trials):
                hidden = hide_ratings(full, p, mechanism)
                global_errors.append(global_mean(hidden) - ref_global)
                user_mean_errors.append(
                    mean_of_nonempty_user_means(hidden) - ref_user
                )

            rows.append({
                "Mechanism": mechanism,
                "Hidden": f"{int(p * 100)}%",
                "Bias of global mean": sum(global_errors) / trials,
                "Random error of global mean": population_std(global_errors),
                "Bias of mean of user means": sum(user_mean_errors) / trials,
                "Random error of mean of user means": population_std(user_mean_errors),
            })
    return rows


experiment_results = missing_data_experiment(full_normal)
headers = list(experiment_results[0].keys())
print(" | ".join(f"{h:>32s}" for h in headers))
for row in experiment_results:
    print(" | ".join(f"{str(row[h]):>32s}" if isinstance(row[h], str)
                     else f"{row[h]:32.6f}" for h in headers))


## Part 6 — Simulate cold start

In [ ]:
def add_cold_start(matrix, dist):
    """Return an expanded copy with 10 new items and 500 new users."""
    old_n_users = len(matrix)
    old_n_items = len(matrix[0])

    # Copy rows and add 10 empty item columns.
    expanded = [row[:] + [None] * 10 for row in matrix]

    # Each new item receives 0–2 ratings from distinct existing users.
    # random.random() chooses the allowed count; random.sample chooses users.
    for new_item_offset in range(10):
        count = int(random.random() * 3)  # 0, 1, or 2
        selected_users = random.sample(range(old_n_users), count)
        ratings = generate_ratings(count, dist)
        new_item_idx = old_n_items + new_item_offset
        for user_idx, rating in zip(selected_users, ratings):
            expanded[user_idx][new_item_idx] = rating

    # Each new user rates 1–3 random old items only.
    for _ in range(500):
        count = 1 + int(random.random() * 3)
        selected_items = random.sample(range(old_n_items), count)
        ratings = generate_ratings(count, dist)
        row = [None] * (old_n_items + 10)
        for item_idx, rating in zip(selected_items, ratings):
            row[item_idx] = rating
        expanded.append(row)

    return expanded


def rated_item_sets(matrix):
    return [set(i for i, rating in enumerate(row) if rating is not None)
            for row in matrix]


def cold_start_metrics(before, after, old_n_users=5000, old_n_items=100):
    after_sets = rated_item_sets(after)
    cold_user_indices = range(old_n_users, len(after))
    cold_item_indices = range(old_n_items, len(after[0]))

    cold_users = sum(len(after_sets[u]) < 5 for u in cold_user_indices)
    cold_items = sum(
        sum(after[u][i] is not None for u in range(len(after))) < 5
        for i in cold_item_indices
    )

    # Average co-rated items for every cold-old user pair.
    cold_old_overlaps = []
    no_overlap_count = 0
    pair_count = 0
    for cold_u in cold_user_indices:
        for old_u in range(old_n_users):
            overlap = len(after_sets[cold_u] & after_sets[old_u])
            cold_old_overlaps.append(overlap)
            pair_count += 1
            if overlap == 0:
                no_overlap_count += 1

    # 1,000 pairs of old users, sampled without replacement from all pairs.
    all_old_pairs = [(u, v) for u in range(old_n_users)
                     for v in range(u + 1, old_n_users)]
    chosen_pair_indices = random.sample(range(len(all_old_pairs)), 1000)
    old_old_overlaps = []
    for idx in chosen_pair_indices:
        u, v = all_old_pairs[idx]
        old_old_overlaps.append(len(after_sets[u] & after_sets[v]))

    return {
        "Sparsity before": sparsity(before),
        "Sparsity after": sparsity(after),
        "Cold users (<5 ratings)": cold_users,
        "Cold items (<5 ratings)": cold_items,
        "Avg co-rated items (cold vs old)": sum(cold_old_overlaps) / len(cold_old_overlaps),
        "Avg co-rated items (old vs old; 1000 pairs)": sum(old_old_overlaps) / len(old_old_overlaps),
        "No-overlap cold-old pairs (%)": 100 * no_overlap_count / pair_count,
    }


uniform_before = matrices["uniform"]
uniform_after = add_cold_start(uniform_before, "uniform")
cold_metrics = cold_start_metrics(uniform_before, uniform_after)

print("Cold-start measurements")
for key, value in cold_metrics.items():
    print(f"{key}: {value:.6f}" if isinstance(value, float) else f"{key}: {value}")


## Part 7 — Discussion questions

In [ ]:
# Print concise, data-grounded discussion prompts and answers.
histogram_spread = {
    dist: normalized_entropy(observed_values(matrices[dist]))
    for dist in DISTRIBUTIONS
}
most_even = max(histogram_spread, key=histogram_spread.get)
normal_counts = [observed_values(matrices["normal"]).count(v) for v in range(1, 6)]
most_common_normal = normal_counts.index(max(normal_counts)) + 1

print("1. Most evenly spread histogram:",
      most_even, "(largest normalized entropy =", histogram_spread[most_even], ").")
print("2. Sparsity is controlled by the number of ratings per user, not rating values.")
print("   Measured sparsities:", {d: metrics[d]["Sparsity"] for d in DISTRIBUTIONS})
print(f"3. In the normal matrix, rating {most_common_normal} appears most often;")
print("   samples are centered around mu=3, and rounding/clipping concentrates values near the center.")
print("4. Average co-rated items (cold vs old):",
      cold_metrics["Avg co-rated items (cold vs old)"])
print("   Cosine similarity cannot be computed from co-rated ratings when overlap is zero.")
print("5. Asking new users to select titles addresses the new-user cold-start problem.")
print("6. Global mean weights each rating; mean of user means weights each nonempty user equally.")
print("   Measured differences:", {d: metrics[d]["Global mean"] - metrics[d]["Mean of user means"]
                                  for d in DISTRIBUTIONS})
print("7. The handout's stated intention is to hide low ratings more often, which would raise observed means.")
print("   However, the literal key u/rating with largest keys retained tends to keep lower ratings,")
print("   so this implementation may show downward bias; see the clarification above.")
print("   Increasing the hidden fraction generally increases random error because fewer ratings remain.")


## Part 8 — GitHub submission
# Put Lab3.ipynb and the three CSV files inside Activity_3/